In [7]:
import os

os.environ["MLFLOW_TRACKING_URI"] = "https://dagshub.com/aakashmohole/wine-quality-e2e-production.mlflow"
os.environ["MLFLOW_TRACKING_USERNAME"] = "aakashmohole"
os.environ["MLFLOW_TRACKING_PASSWORD"] = "966e68bc3e9374ca23d7ff1e223912705356fec3"

In [8]:
import os

# safely change directory to project root
if os.path.split(os.getcwd())[-1] == "research":
    os.chdir("../")

%pwd

'/Users/apple/Aakash/Projects/wine-quality-e2e-production'

In [9]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class ModelEvaluationConfig:
    root_dir: Path
    test_data_path: Path
    model_path: Path
    all_params: dict
    target_column:str
    mlflow_uri:str
    metric_file_name : str


In [10]:
from src.wine_quality_project.constants import *
from src.wine_quality_project.utils.common import read_yaml, create_directories, save_json

class ConfigrationManager:
    def __init__(self,
                 config_filepath= CONFIG_FILE_PATH,
                 params_filepath = PARAMS_FILE_PATH,
                 schema_filepath = SCHEMA_FILE_PATH
                 ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_model_evaluation_config(self) -> ModelEvaluationConfig:
        config = self.config.model_evaluation
        params = self.params.ElasticNet
        schema = self.schema.TARGET_COLUMN

        create_directories([config.root_dir])

        model_evaluation_config=ModelEvaluationConfig(
            root_dir=config.root_dir,
            test_data_path=config.test_data_path,
            model_path=config.model_path,
            all_params=params,
            metric_file_name=config.metric_file_name,
            target_column=schema.name,
            mlflow_uri="https://dagshub.com/aakashmohole/wine-quality-e2e-production.mlflow"
        )
        return model_evaluation_config

In [15]:
import os
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from src.wine_quality_project.utils.common import save_json

from urllib.parse import urlparse
import mlflow
import mlflow.sklearn
import numpy as np
import joblib

class ModelEvaluation:
    def __init__(self, config: ModelEvaluationConfig):
        self.config = config
        
    def eval_metrics(self, actual, pred):
        rmse = np.sqrt(mean_squared_error(actual,pred))
        mae = mean_absolute_error(actual, pred)
        r2 = r2_score(actual, pred)

        return rmse, mae, r2
    
    def log_into_mlflow(self):
        test_data = pd.read_csv(self.config.test_data_path)
        model = joblib.load(self.config.model_path)
        
        test_x = test_data.drop([self.config.target_column], axis=1)
        test_y = test_data[[self.config.target_column]]

        mlflow.set_registry_uri(self.config.mlflow_uri)
        tracking_uri_type_store = urlparse(mlflow.get_tracking_uri()).scheme

        with mlflow.start_run():

            predicted_qualities = model.predict(test_x)

            rmse, mae, r2 = self.eval_metrics(test_y, predicted_qualities)

            # save metrics as locals
            score = {"rmse":rmse, "mae": mae, "r2": r2}

            save_json(path=Path(self.config.metric_file_name), data= score)

            mlflow.log_params(self.config.all_params)
            mlflow.log_metric("rmse", rmse)
            mlflow.log_metric("mae", mae)
            mlflow.log_metric("r2", r2)

            if tracking_uri_type_store != "file":
                mlflow.sklearn.log_model(model, "model", registered_model_name="ElasticNetModel")
            else:
                mlflow.sklearn.log_model(model, "model")


In [16]:
try:
    config = ConfigrationManager()
    model_evaluation_config = config.get_model_evaluation_config()
    model_evaluation = ModelEvaluation(config=model_evaluation_config)
    model_evaluation.log_into_mlflow()
except Exception as e:
    raise e


[2026-05-09 22:18:25,999: INFO: common: yaml file: config/config.yaml loaded successfully]
[2026-05-09 22:18:26,002: INFO: common: yaml file: params.yaml loaded successfully]
[2026-05-09 22:18:26,005: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-05-09 22:18:26,007: INFO: common: created directory at: artifacts]
[2026-05-09 22:18:26,008: INFO: common: created directory at: artifacts/model_evaluation]
[2026-05-09 22:18:26,619: INFO: common: json file saved at: artifacts/model_evaluation/metrics.json]


2026/05/09 22:18:30 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/09 22:18:40 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
Successfully registered model 'ElasticNetModel'.
2026/05/09 22:18:44 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: ElasticNetModel, version 1
Created version '1' of model 'ElasticNetModel'.


🏃 View run fortunate-squid-672 at: https://dagshub.com/aakashmohole/wine-quality-e2e-production.mlflow/#/experiments/0/runs/14e600fa128c49dc95cce501b811f25b
🧪 View experiment at: https://dagshub.com/aakashmohole/wine-quality-e2e-production.mlflow/#/experiments/0
